In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/archive.zip'
extract_path = '/content/chest_xray'

if not os.path.exists(extract_path):
    print("Unzipping dataset... this may take a minute.")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Done unzipping!")
else:
    print("Dataset already unzipped. Skipping.")

Unzipping dataset... this may take a minute.
Done unzipping!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
DATA_DIR = '/content/chest_xray/chest_xray/'

In [ ]:
BATCH_SIZE=32
EPOCHS=40
LEARNING_RATE=0.001

In [ ]:
# Transformation of images (MobileNetV2 needs 224*224)
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
val_test_transform= transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   transform=val_test_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'),  transform=val_test_transform)

In [ ]:
#Dataloaders break the dataset into batches and shuffle training data
train_loader= torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
print(f"Training images  : {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Test images  : {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

Training images  : 5216
Validation images: 16
Test images  : 624
Classes: ['NORMAL', 'PNEUMONIA']


In [ ]:
# Loading the Pretrained Mobilenetv2
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 84.9MB/s]


In [ ]:
# Freeze all the base layers (feature extractor) of MobileNetV2
for param in model.features.parameters():
    param.requires_grad = False


In [ ]:
# replacing the final classification layer (original is 1000 classes) with 2 classes (normal and pneumonia)
num_features= model.classifier[1].in_features # original inputs from the pretrained model but I have changed it to provide only two output classes
model.classifier[1] = nn.Linear(num_features, 2)

In [ ]:
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
# Optimizer take the gradient and update weights in backpropagation
optimizer = optim.SGD(model.classifier.parameters(), lr=LEARNING_RATE, momentum=0.9)


In [ ]:
# Cosine Annealing Scheduler gently drops the learning rate following a cosine curve
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# Inside training loop
  # For each epoch -1) Train on training data ( weights are updated)
  #                 2) Validate on validation data(weights are not updated)

In [ ]:
train_losses=[]
val_losses=[]

In [ ]:
for epoch in range(EPOCHS):
  print(f"\n---Epoch {epoch+1} / {EPOCHS}---")

  model.train()
  running_trainin_loss=0.0

  for images , labels in train_loader:
      images = images.to(device)   # Move images to GPU
      labels = labels.to(device)   # Move labels to GPU

      optimizer.zero_grad() # for clearing old gradients
      outputs =model(images) #forward pass
      loss =criterion(outputs,labels) #calculating training loss
      loss.backward() # backpropagation of calculated gradient
      optimizer.step() #updating the weights

      running_trainin_loss+=loss.item()

  avg_train_loss=running_trainin_loss/len(train_loader)
  train_losses.append(avg_train_loss)
  print(f"Training Loss: {avg_train_loss:.4f}")

  #Validation Phase
  model.eval()
  running_val_loss=0.0
  correct=0
  total=0

  with torch.no_grad(): # we dont need gradient calculation here so disabling it
     for images, labels in test_loader: # I am changing to validate on test set
         images = images.to(device)
         labels = labels.to(device)

         outputs=model(images)
         loss=criterion(outputs,labels)
         running_val_loss+=loss.item()

         _,predicted=torch.max(outputs.data,1)
         total+=labels.size(0)
         correct+=(predicted==labels).sum().item()

  avg_val_loss=running_val_loss/len(test_loader)
  val_losses.append(avg_val_loss)
  accuracy=100*correct/total
  print(f"Training Loss: {avg_train_loss:.4f}")
  print(f"Validation Loss: {avg_val_loss:.4f}")
  print(f"Validation Accuracy: {accuracy:.2f}%")

# Step the scheduler at the end of each epoch to update the learning rate
  scheduler.step()



---Epoch 1 / 40---
Training Loss: 0.3961
Training Loss: 0.3961
Validation Loss: 0.4708
Validation Accuracy: 74.52%

---Epoch 2 / 40---
Training Loss: 0.2690
Training Loss: 0.2690
Validation Loss: 0.4050
Validation Accuracy: 79.33%

---Epoch 3 / 40---


In [ ]:
# Plotting the Training Loss and Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss',   marker='o', color='blue')
plt.plot(range(1, EPOCHS + 1), val_losses,   label='Validation Loss', marker='o', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss per Epoch')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('loss_plot.png')
plt.show()
print("Plot saved as loss_plot.png")


In [ ]:
# Final test on the Test Dataset

print("----Final test on the Test Dataset----")
model.eval()
correct=0
total=0

with torch.no_grad():
  for images,labels in test_loader:
    images=images.to(device)
    labels=labels.to(device)
    outputs=model(images)
    _,predicted=torch.max(outputs.data,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()

  print('Accuracy of the network on the test images: {} %'.format(100*correct /total))

In [ ]:
#Saving the trained model
torch.save(model.state_dict(), 'mobilenetv2_pneumonia.pth')
print("Model saved as mobilenetv2_pneumonia.pth")
